# Dynamical CBED: Many-Beam Coupling, Absorption, HOLZ Lines, and the Point Group

Tutorial 28 built a convergent-beam pattern out of independent two-beam discs. That is enough
for disc geometry, for the Kossel–Möllenstedt threshold, and for the thickness analysis, and it
is honest about what it is not enough for. Its `describe()` said so in as many words: no
many-beam coupling, no absorption, no HOLZ lines, and no diffraction-group symmetry
determination.

This tutorial closes all four. They are not four separate features:

1. **Many-beam coupling** makes the pattern one object instead of a collection of independent
   calculations. At a zone axis dozens of reflections are simultaneously near the Bragg
   condition and exchange intensity; the two-beam premise is at its worst exactly where CBED is
   done.
2. **Absorption** — an imaginary part of the crystal potential — makes the result physically
   realizable. Without it fringes never decay and the bright-field disc is symmetric in a way no
   real one is.
3. **HOLZ lines** appear when higher-order Laue zone reflections join the beam set. They are the
   sharp features that measure a lattice parameter to a part in ten thousand.
4. **The point group, centre included.** Friedel's law makes $I_g = I_{-g}$ in kinematic
   diffraction whatever the structure, so a selected-area pattern stops at the Laue class: 11
   possibilities where there are 32. CBED gets past it. This is the technique's most celebrated
   capability, and the mechanism turns out to be item 3: admitting HOLZ beams is what breaks the
   projection symmetry that would otherwise make every pattern look centrosymmetric.

Everything below is computed live. The theory is
`docs/site/theory/dynamical_cbed_and_symmetry_determination.md`; the modules are
`pytex.diffraction.dynamical`, `pytex.diffraction.holz` and
`pytex.diffraction.diffraction_groups`.


## 0. Setup


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from pytex import (
    AbsorptionModel,
    AtomicSite,
    ConvergentBeamConfig,
    Lattice,
    Phase,
    SpaceGroupSpec,
    SymmetryObservations,
    SymmetrySpec,
    UnitCell,
    ZoneAxis,
    beam_set_for_zone,
    beam_set_from_indices,
    crystal_frame,
    determine_point_group,
    diffraction_group_for,
    diffraction_group_symbols,
    diffraction_group_table,
    extinction_distance_angstrom,
    holz_line_pattern,
    potential_coefficients_inv_angstrom,
    simulate_cbed_pattern,
    solve_bloch_waves,
    structure_matrix,
    two_beam_rocking_curve,
)
from pytex.core.point_groups import PointGroup, all_point_group_symbols
from pytex.diffraction.kinematic import electron_wavelength_angstrom

CRYSTAL = crystal_frame()
BEAM_KEV = 200.0
LAMBDA = electron_wavelength_angstrom(BEAM_KEV)

FCC_BASIS = ((0.0, 0.0, 0.0), (0.0, 0.5, 0.5), (0.5, 0.0, 0.5), (0.5, 0.5, 0.0))


def fcc_derived(name, parameter, sublattices, point_group, space_group):
    # One FCC sublattice per (species, offset) pair, in the conventional cubic cell.
    lattice = Lattice(parameter, parameter, parameter, 90.0, 90.0, 90.0, crystal_frame=CRYSTAL)
    sites = tuple(
        AtomicSite(
            label=f"{species}{index}",
            species=species,
            fractional_coordinates=np.asarray(base, dtype=float) + np.asarray(offset, dtype=float),
        )
        for species, offset in sublattices
        for index, base in enumerate(FCC_BASIS)
    )
    return Phase(
        name,
        lattice=lattice,
        symmetry=SymmetrySpec.from_point_group(point_group, reference_frame=CRYSTAL),
        crystal_frame=CRYSTAL,
        unit_cell=UnitCell(lattice=lattice, sites=sites),
        space_group=SpaceGroupSpec(symbol=space_group[0], number=space_group[1],
                                   reference_frame=CRYSTAL),
    )


NICKEL = fcc_derived("nickel-fcc", 3.5239, (("Ni", (0, 0, 0)),), "m-3m", ("Fm-3m", 225))

# The controlled pair: zincblende and diamond. Identical structure type, identical
# site geometry, eight atoms per conventional cell in both. The *only* difference is
# whether the two sublattices carry the same element -- and that difference is
# precisely a centre of symmetry.
GAAS = fcc_derived("gallium-arsenide", 5.6535,
                   (("Ga", (0, 0, 0)), ("As", (0.25, 0.25, 0.25))), "-43m", ("F-43m", 216))
SILICON = fcc_derived("silicon", 5.4310,
                      (("Si", (0, 0, 0)), ("Si", (0.25, 0.25, 0.25))), "m-3m", ("Fd-3m", 227))

for phase in (NICKEL, GAAS, SILICON):
    group = PointGroup.from_symbol(phase.symmetry.point_group)
    print(f"{phase.name:<20} a = {phase.lattice.a:.4f} A  point group {group.hermann_mauguin:<5} "
          f"centrosymmetric: {group.is_centrosymmetric}")
print(f"\nwavelength at {BEAM_KEV:.0f} kV: {LAMBDA:.5f} A")


## 1. The coupled beam equations

Inside the crystal the wavefield is a sum over reciprocal-lattice beams, and in the column
approximation their amplitudes obey one linear system:

$$\frac{\mathrm{d}\psi}{\mathrm{d}z} = i\pi\,\mathbf{A}\,\psi,
\qquad A_{gg} = 2 s_g + \frac{i}{\xi'_0},
\qquad A_{gh} = \nu_{g-h} + \frac{i}{\xi'_{g-h}} \;\; (g \neq h).$$

The off-diagonal coupling is the Fourier coefficient of the scaled lattice potential,

$$\nu_g = \frac{\lambda F_g}{\pi V_c \cos\theta_g}, \qquad |\nu_g| = \frac{1}{\xi_g},$$

so its **modulus** is the reciprocal of the extinction distance tutorial 28 validated against
Williams and Carter's Table 23.1. The many-beam module does not introduce a second absolute
scale; it inherits the one that was already checked. Its **phase**, on the other hand, is new,
and section 5 is entirely about what that phase carries.

$\mathbf{A}$ does not depend on depth, so the solution is a single matrix exponential, evaluated
by eigen-decomposition:

$$\psi(t) = \exp(i\pi\mathbf{A}t)\,\mathbf{e}_0
  \quad\Longrightarrow\quad
  \psi_g(t) = \sum_j C_{gj}\,\alpha_j\,e^{i\pi\gamma_j t},
  \qquad \boldsymbol{\alpha} = \mathbf{C}^{-1}\mathbf{e}_0 .$$

The eigenvectors are the **Bloch waves** — wavefields that pass through the crystal unchanged in
shape, each attenuated at its own rate $\mathrm{Im}\,\gamma_j$. The eigenvalues trace the
dispersion surface.


In [ ]:
zone_001_ni = ZoneAxis(np.array([0, 0, 1]), phase=NICKEL)
beams = beam_set_for_zone(NICKEL, zone_001_ni, beam_energy_kev=BEAM_KEV,
                          convergence_semi_angle_mrad=6.0)
print(beams.describe())
print()

A = structure_matrix(beams, [[0.0, 0.0]])[0]
print(f"structure matrix at zero tilt: {A.shape[0]} x {A.shape[1]}")
print(f"  Hermitian (real potential):        {np.allclose(A, A.conj().T)}")
print(f"  symmetric (centrosymmetric):       {np.allclose(A, A.T)}")
print(f"  |nu_220| = {abs(A[0, beams.index_of([2, 2, 0])]):.6f} 1/A")
print(f"  1/xi_220 = {1.0 / float(extinction_distance_angstrom(NICKEL, [[2, 2, 0]])[0]):.6f} 1/A")


### 1.1 The two-beam limit is exact

The first thing to demand of a many-beam solver is that it reproduce the closed form when there
are only two beams. That single check pins three conventions at once — the diagonal $2 s_g$, the
off-diagonal scale, and the $i\pi$ in the propagator. Any one of them wrong gives a rocking
curve of the right general shape and the wrong fringe spacing, which is exactly the kind of
error that survives a plausibility check.


In [ ]:
two_beam = beam_set_from_indices(NICKEL, zone_001_ni, [[2, 2, 0]], beam_energy_kev=BEAM_KEV)

# Walk along the in-plane direction of g: the excitation error is affine in the tilt,
# so this sweeps s linearly and can be inverted for the tilts we want.
g_zone = two_beam.g_zone[1]
in_plane = float(np.linalg.norm(g_zone[:2]))
zero_tilt_s = g_zone[2] - 0.5 * LAMBDA * two_beam.g_magnitude_inv_angstrom[1] ** 2
targets = np.linspace(-0.02, 0.02, 401)
tilts = ((zero_tilt_s - targets) / in_plane)[:, None] * (g_zone[:2] / in_plane)[None, :]

THICKNESS = 800.0
solution = solve_bloch_waves(two_beam, tilts, thickness_angstrom=THICKNESS)
closed_form = two_beam_rocking_curve(
    targets, thickness_angstrom=THICKNESS,
    extinction_distance_angstrom=float(extinction_distance_angstrom(NICKEL, [[2, 2, 0]])[0]),
)
print(f"max |Bloch - closed form| over 401 points: "
      f"{np.max(np.abs(solution.intensity_of([2, 2, 0]) - closed_form)):.3e}")
print(f"max |sum of beams - 1|                   : "
      f"{np.max(np.abs(solution.total_intensity - 1.0)):.3e}")


The agreement is at machine precision, and the second number is the other exact property worth
having: without absorption $\mathbf{A}$ is Hermitian, so the propagator is unitary and the beam
intensities sum to one at *every* thickness and *every* incident direction. That identity is the
only global check available on a many-beam calculation, and it is the one that catches the
classic implementation error — obtaining the excitation amplitudes $\boldsymbol{\alpha}$ by
projection, $\alpha_j = C_{0j}^*$, which is only valid if the eigenvectors are orthogonal. They
are not.


### 1.2 What the coupling actually changes

Now the same reflection with the rest of the zone present.


In [ ]:
many = solve_bloch_waves(beams, tilts, thickness_angstrom=THICKNESS)
print(f"beams in the coupled calculation: {beams.size}")
print(f"sum over beams stays unity to    : {np.max(np.abs(many.total_intensity - 1.0)):.2e}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(targets * 1e3, closed_form, label="two beams (closed form)", lw=1.6)
axes[0].plot(targets * 1e3, many.intensity_of([2, 2, 0]), label=f"{beams.size} beams", lw=1.2)
axes[0].set_xlabel(r"$s_{220}$  (1e-3 $\AA^{-1}$)")
axes[0].set_ylabel("diffracted intensity")
axes[0].set_title(f"(220) of nickel, t = {THICKNESS:.0f} A")
axes[0].legend(fontsize=8)

axes[1].plot(targets * 1e3, 1.0 - closed_form, label="two-beam complement", lw=1.6)
axes[1].plot(targets * 1e3, many.transmitted_intensity, label="many-beam bright field", lw=1.2)
axes[1].set_xlabel(r"$s_{220}$  (1e-3 $\AA^{-1}$)")
axes[1].set_ylabel("transmitted intensity")
axes[1].set_title("the direct beam")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()


The diffracted curve keeps its fringe *positions* — those are set by the thickness and the
extinction distance — but not its amplitude, because the reflection now shares intensity with
its neighbours. The transmitted beam is the sharper lesson: in the two-beam picture it is
*defined* as $1 - I_g$, so it carries no independent information. In the coupled calculation it
is a genuine bright-field intensity, and it is what every disc-symmetry argument in section 5
will be read from.


### 1.3 The dispersion surface

The eigenvalues $\gamma_j$ are the Bloch-wave excitations. Their real parts are the dispersion
surface — the branches whose gaps at the Bragg conditions are the dynamical effect itself.


In [ ]:
kept = solve_bloch_waves(beams, tilts, thickness_angstrom=THICKNESS, keep_eigenbasis=True)
branches = np.sort(np.real(kept.eigenvalues), axis=1)

fig, ax = plt.subplots(figsize=(6.4, 3.6))
for column in range(branches.shape[1]):
    ax.plot(targets * 1e3, branches[:, column] * 1e3, lw=0.9)
ax.set_xlabel(r"$s_{220}$  (1e-3 $\AA^{-1}$)")
ax.set_ylabel(r"$\mathrm{Re}\,\gamma_j$  (1e-3 $\AA^{-1}$)")
ax.set_title(f"dispersion surface, {beams.size} branches")
fig.tight_layout()
plt.show()

# Storing the eigenbasis makes a thickness series one matrix product per thickness
# instead of a new eigen-decomposition.
thicknesses = np.linspace(100.0, 3000.0, 60)
series = kept.intensities_at(thicknesses)
centre = targets.size // 2
print(f"thickness series from the stored eigenbasis: {series.shape} (thickness, tilt, beam)")


## 2. Absorption

Electrons leave the coherent elastic wavefield by processes the calculation does not follow —
thermal diffuse scattering above all. The standard representation adds an imaginary part to
every Fourier coefficient, $\xi_g^{-1} \to \xi_g^{-1} + i\,\xi_g'^{-1}$.

**What is derived and what is assumed** is worth stating precisely, because the two are usually
run together. The *structure* is not an approximation: an imaginary optical potential is the
correct way to represent loss from a coherent wavefield, and every observable consequence below
follows from the eigenvector structure rather than being applied to the output afterwards. The
*magnitudes* are phenomenological — `AbsorptionModel` carries them as ratios with the customary
working value $0.1$ of Hirsch *et al.*, and says so. First-principles absorptive form factors are
not implemented.


In [ ]:
print(AbsorptionModel().describe())
print()
print(AbsorptionModel.none().describe())


### 2.1 Normal absorption is exactly a scalar

The mean absorptive coefficient $i/\xi'_0$ sits on *every* diagonal element, so it commutes out
of the matrix exponential:

$$\exp\!\left[i\pi\left(\mathbf{A} + \tfrac{i}{\xi'_0}\mathbf{I}\right)t\right]
  = e^{-\pi t/\xi'_0}\exp(i\pi\mathbf{A}t),
  \qquad I_g \to e^{-2\pi t/\xi'_0}\,I_g .$$

It can therefore change no relative intensity, no fringe position, and no symmetry — which means
the phenomenological number cannot contaminate any conclusion drawn below. That is a claim worth
checking rather than asserting.


In [ ]:
normal_only = solve_bloch_waves(beams, tilts, thickness_angstrom=900.0,
                                absorption=AbsorptionModel(mean_ratio=0.12, reflection_ratio=0.0))
free = solve_bloch_waves(beams, tilts, thickness_angstrom=900.0)
factor = normal_only.normal_absorption_factor
print(f"exp(-2 pi t / xi'_0) = {factor:.6f}")
print(f"max |I_absorbed / factor - I_free| = "
      f"{np.max(np.abs(normal_only.intensities / factor - free.intensities)):.3e}")


### 2.2 Anomalous absorption: the bright field goes asymmetric, the dark field does not

The two Bloch waves of a two-beam calculation have their intensity maxima *on* and *between* the
atomic planes. The first is absorbed strongly and the second weakly, and which one is
preferentially excited depends on the sign of $s$. The consequence is the Hashimoto–Howie–Whelan
result, and it is a much better test of an absorption implementation than any single number,
because it is a property no incorrect implementation reproduces by accident.


In [ ]:
wide = np.linspace(-0.03, 0.03, 601)
wide_tilts = ((zero_tilt_s - wide) / in_plane)[:, None] * (g_zone[:2] / in_plane)[None, :]
absorbed = solve_bloch_waves(two_beam, wide_tilts, thickness_angstrom=1500.0,
                             absorption=AbsorptionModel())
elastic = solve_bloch_waves(two_beam, wide_tilts, thickness_angstrom=1500.0)

bright = absorbed.transmitted_intensity
dark = absorbed.intensity_of([2, 2, 0])
print(f"dark-field asymmetry, with absorption   : "
      f"{np.max(np.abs(dark - dark[::-1])) / dark.max():.2e}")
print(f"bright-field asymmetry, with absorption : "
      f"{np.max(np.abs(bright - bright[::-1])) / bright.max():.3f}")
print(f"bright-field asymmetry, without         : "
      f"{np.max(np.abs(elastic.transmitted_intensity - elastic.transmitted_intensity[::-1])):.2e}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True)
axes[0].plot(wide * 1e3, elastic.transmitted_intensity, label="no absorption", lw=1.4)
axes[0].plot(wide * 1e3, bright / bright.max(), label="absorbing (rescaled)", lw=1.4)
axes[0].axvline(0.0, color="0.6", lw=0.8, ls=":")
axes[0].set_title("bright field: asymmetric about $s=0$")
axes[1].plot(wide * 1e3, elastic.intensity_of([2, 2, 0]), label="no absorption", lw=1.4)
axes[1].plot(wide * 1e3, dark / dark.max(), label="absorbing (rescaled)", lw=1.4)
axes[1].axvline(0.0, color="0.6", lw=0.8, ls=":")
axes[1].set_title("dark field: still symmetric")
for axis in axes:
    axis.set_xlabel(r"$s_{220}$  (1e-3 $\AA^{-1}$)")
    axis.legend(fontsize=8)
axes[0].set_ylabel("intensity (normalized)")
fig.tight_layout()
plt.show()


### 2.3 And the fringes finally decay


In [ ]:
depths = np.linspace(50.0, 4000.0, 300)
at_bragg = ((zero_tilt_s - 0.0) / in_plane) * (g_zone[:2] / in_plane)
kept_two = solve_bloch_waves(two_beam, at_bragg[None, :], thickness_angstrom=100.0,
                             absorption=AbsorptionModel(), keep_eigenbasis=True)
kept_free = solve_bloch_waves(two_beam, at_bragg[None, :], thickness_angstrom=100.0,
                              keep_eigenbasis=True)

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(depths, kept_free.intensities_at(depths)[:, 0, 1], lw=1.2, label="no absorption")
ax.plot(depths, kept_two.intensities_at(depths)[:, 0, 1], lw=1.4, label="absorbing")
ax.set_xlabel("foil thickness (A)")
ax.set_ylabel("$I_{220}$ at exact Bragg")
ax.set_title("Pendelloesung fringes, with and without absorption")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()


## 3. HOLZ lines

A zone-axis pattern is blind to the lattice repeat along the beam and nearly blind to small
changes in the other two: a spot moves by the same fractional amount the lattice does, which for
a strain of $10^{-4}$ is nothing. Higher-order Laue zone lines are the way out.

Because $s_g$ is affine in the incident tilt, the locus $s_g = 0$ is a **straight line** in the
plane of incident directions:

$$\boldsymbol{\theta}\cdot\hat{\mathbf{g}}_\perp = d_g,
\qquad d_g = \frac{g_z - \tfrac12\lambda|\mathbf{g}|^2}{|\mathbf{g}_\perp|}.$$

It is dark (*deficiency*) in the direct disc and bright (*excess*) at the same incident tilts in
the disc of $\mathbf{g}$ itself. Nothing is approximated beyond $s_g$ itself, so the positions
are exact — and PyTex checks them by asking the *dynamical* module, which derived $s_g$ for a
different purpose, what the excitation error is at points on the line.


In [ ]:
lines = holz_line_pattern(NICKEL, zone_001_ni, beam_energy_kev=BEAM_KEV,
                          convergence_semi_angle_mrad=8.0, max_index=24,
                          g_max_inv_angstrom=6.0)
print(lines.describe())
print()

# The independent check: s_g must vanish along the line.
worst = 0.0
for line in lines.lines[:8]:
    chord = line.chord_tilt_rad()
    probe = beam_set_from_indices(NICKEL, zone_001_ni, line.miller_indices[None, :],
                                  beam_energy_kev=BEAM_KEV)
    samples = np.stack([chord[0], chord.mean(axis=0), chord[1]])
    worst = max(worst, float(np.max(np.abs(probe.excitation_errors(samples)[:, 1]))))
print(f"max |s_g| on the line, from pytex.diffraction.dynamical: {worst:.2e} 1/A")


In [ ]:
alpha_mrad = lines.convergence_semi_angle_mrad
fig, ax = plt.subplots(figsize=(5.4, 5.4))
circle = plt.Circle((0, 0), alpha_mrad, fill=False, color="0.3", lw=1.4)
ax.add_patch(circle)
for line in lines.bright_field_lines:
    chord = line.chord_tilt_rad() * 1e3
    ax.plot(chord[:, 0], chord[:, 1], lw=0.7, color="C0", alpha=0.8)
ax.set_xlim(-1.15 * alpha_mrad, 1.15 * alpha_mrad)
ax.set_ylim(-1.15 * alpha_mrad, 1.15 * alpha_mrad)
ax.set_aspect("equal")
ax.set_xlabel(r"$\theta_u$ (mrad)")
ax.set_ylabel(r"$\theta_v$ (mrad)")
ax.set_title(f"first-order HOLZ deficiency lines, nickel [001]\n"
             f"{len(lines.bright_field_lines)} lines inside a {alpha_mrad:.0f} mrad disc")
fig.tight_layout()
plt.show()


### 3.1 Sharpness, sensitivity, and why intersections are what gets measured

The rocking curve's first zero is at $|s| = 1/t$, and $s$ varies across the disc at the rate
$|\mathbf{g}_\perp|$ per radian, so the line's angular half-width is
$\Delta\theta = 1/(t\,|\mathbf{g}_\perp|)$. It narrows in proportion to the foil thickness: HOLZ
metrology wants a *thick* specimen, which is the opposite of the usual thin-foil instinct.

The lever arm is $\partial d_g/\partial\varepsilon = \lambda|\mathbf{g}|^2/(2|\mathbf{g}_\perp|)$
per unit of isotropic lattice strain. Divide one by the other and you get the strain that moves a
line by its own width.


In [ ]:
print(f"  {'hkl':<14} {'|g_perp|':>9} {'d (mrad)':>9} {'dd/deps':>9} {'width':>9} {'eps_res':>10}")
for line in lines.bright_field_lines[:6]:
    print(f"  {str(tuple(int(v) for v in line.miller_indices)):<14} "
          f"{line.g_perp_inv_angstrom:9.3f} {line.offset_rad * 1e3:9.3f} "
          f"{line.strain_sensitivity_rad * 1e3:9.2f} "
          f"{line.angular_width_rad(1000.0) * 1e3:9.4f} "
          f"{line.resolvable_strain(1000.0):10.2e}")

best_line = min(lines.bright_field_lines, key=lambda item: item.resolvable_strain(1000.0))
crossings = lines.intersections()
best_cross = crossings[0]
print(f"\nbest single line       : resolvable strain {best_line.resolvable_strain(1000.0):.2e}")
print(f"best line intersection : {tuple(int(v) for v in best_cross.first_indices)} x "
      f"{tuple(int(v) for v in best_cross.second_indices)} at {best_cross.angle_deg:.1f} deg, "
      f"moving {best_cross.strain_sensitivity_rad * 1e3:.0f} mrad per unit strain")
print(f"                         resolvable strain "
      f"{best_line.angular_width_rad(1000.0) / best_cross.strain_sensitivity_rad:.2e}")


There it is: a single line resolves a strain of a few times $10^{-3}$, which is nowhere near the
$10^{-4}$ the technique is famous for. A crossing of two *nearly parallel* lines moves as
$1/\sin\phi$ times faster and gets there. That is why HOLZ measurements read intersections rather
than individual lines, and `intersections()` reports the amplification instead of leaving it to
be rediscovered.


### 3.2 The trap: strain and voltage are exactly degenerate

Scale the lattice by $1+\varepsilon$ and every $\mathbf{g}$ shrinks by the same factor:

$$d_g(\varepsilon, \lambda) = \frac{g_z}{|\mathbf{g}_\perp|}
  - \frac{\lambda|\mathbf{g}|^2}{2(1+\varepsilon)|\mathbf{g}_\perp|}.$$

The wavelength enters the *same* term with the opposite sign. So a fractional change in lattice
parameter and a fractional change in wavelength shift every line by exactly compensating amounts,
simultaneously, at every reflection. Nothing in the pattern can separate them.


In [ ]:
strain = 1e-3
residual = max(
    abs(line.offset_at(lattice_strain=strain, wavelength_angstrom=LAMBDA * (1.0 + strain))
        - line.offset_rad)
    for line in lines.lines
)
print(f"strain {strain:.0e} compensated by the same fractional wavelength change:")
print(f"  max |offset shift| over {len(lines.lines)} lines = {residual:.2e} rad")
print()
shifted = [line.offset_at(lattice_strain=strain) - line.offset_rad for line in lines.lines[:4]]
print("the same strain with the wavelength held fixed shifts the first four lines by")
print("  " + ", ".join(f"{value * 1e3:+.4f} mrad" for value in shifted))


This is not a limitation of the model. It is why quantitative HOLZ metrology begins by
calibrating the accelerating voltage against a standard of known lattice parameter — and why a
lattice parameter quoted from an uncalibrated microscope is really a statement about its
high-tension supply.


## 4. Diffraction groups

Now the symmetry. Fix a beam direction $\mathbf{b}$; each operator $S$ of the crystal point group
falls into one of three cases.

- $S\mathbf{b} = \mathbf{b}$: a genuine symmetry of the experiment. It contributes its
  restriction $T = S|_\perp$ to the plane normal to the beam, **untagged**.
- $S\mathbf{b} = -\mathbf{b}$: it reverses the beam, so it is not a symmetry on its own —
  combined with the reciprocity theorem it becomes one. It contributes $T = S|_\perp$ **tagged**,
  written with the subscript $R$.
- Neither: it relates different patterns and contributes nothing.

That map is a homomorphism onto a subgroup of $G_2 \times \mathbb{Z}_2$ with $G_2$ one of the ten
plane point groups. Enumerating the possibilities gives 10 with no tagged element, 10 direct
products (suffix $1_R$), and 11 graphs of a surjection onto $\mathbb{Z}_2$ — **31**, which is
Buxton, Eades, Steeds and Rackham's count. PyTex derives them rather than storing them, so the
number below is a result and not a transcription.


In [ ]:
symbols = diffraction_group_symbols()
print(f"diffraction groups derived from PyTex's own operator tables: {len(symbols)}")
print(", ".join(symbols))


### 4.1 What each group predicts

Two observables follow from the element list alone.

**Whole pattern.** A tagged element needs reciprocity, which relates a point in one disc to a
point in another at an incident direction *outside* the illumination cone. Only untagged elements
are rigid symmetries of the recorded pattern.

**Bright field.** Inside the direct disc the reciprocity displacement is proportional to
$\mathbf{g}_\perp$ and therefore vanishes, leaving only reciprocity's own inversion of the
incident direction. So a tagged element acts there as $-T$ rather than $T$:
$\mathrm{BF} = \varphi(D)$ with $\varphi(T,\text{tagged}) = -T$.

The familiar consequences drop out. A two-fold axis across the beam gives a bright-field mirror
the whole pattern does not have; a mirror perpendicular to the beam gives a bright-field two-fold
alone; and $\bar{6}m2$ down its three-fold shows a *six*-fold direct disc over a $3m$ pattern.


In [ ]:
cases = [("m-3m", (0, 0, 1)), ("m-3m", (1, 1, 1)), ("432", (0, 0, 1)),
         ("-43m", (0, 0, 1)), ("-43m", (1, 1, 1)), ("-6m2", (0, 0, 1)),
         ("2", (1, 0, 0)), ("m", (0, 0, 1)), ("-1", (0, 0, 1))]
print(f"  {'point group':<12} {'beam':<10} {'diffraction group':<12} {'BF':<5} {'WP':<5} {'2_R'}")
for symbol, direction in cases:
    group = diffraction_group_for(symbol, direction)
    print(f"  {symbol:<12} {str(direction):<10} {group.symbol:<12} "
          f"{group.bright_field_symbol:<5} {group.whole_pattern_symbol:<5} "
          f"{group.has_friedel_symmetry}")


### 4.2 The centre of symmetry, as a theorem

The element $2_R$ needs an operator acting as $-1$ on the beam direction **and** as $-1$ on the
transverse plane. That is the inversion, and nothing else. So

$$2_R \in D \text{ at any beam direction} \iff \text{the crystal is centrosymmetric},$$

which is an exact correspondence rather than a tendency. Checking it over all 32 point groups at
every characteristic direction takes one cell.


In [ ]:
violations = []
for symbol in all_point_group_symbols():
    centric = PointGroup.from_symbol(symbol).is_centrosymmetric
    for entry in diffraction_group_table(symbol):
        if entry.diffraction_group.has_friedel_symmetry is not centric:
            violations.append((symbol, entry.diffraction_group.symbol))
print(f"point groups scanned          : {len(all_point_group_symbols())}")
print(f"beam directions with 2_R wrong: {len(violations)}")

acentric = determine_point_group(SymmetryObservations(friedel_pair_two_fold=False))
centric = determine_point_group(SymmetryObservations(friedel_pair_two_fold=True))
print(f"\nknowing only the +-g relation splits the 32 point groups into "
      f"{len(acentric.point_groups)} acentric and {len(centric.point_groups)} centric.")
print("That is the arithmetic of the whole technique: selected-area diffraction stops at the")
print("11 Laue classes, and this is the distinction Friedel's law destroyed.")


Note that $2_R$ is invisible in *both* disc symmetries — $\varphi(2, \text{tagged}) = -2 = 1$ —
which is why the $\pm\mathbf{g}$ observation is a separate field on `SymmetryObservations` and
why leaving it unknown leaves the verdict open by construction. Section 5.3 says why PyTex does
not try to measure it from a zone-axis simulation.


### 4.3 Choosing the zone axis

Two point groups that share a diffraction group at one beam direction generally differ at
another. `diffraction_group_table` is what an experiment plan is built from.


In [ ]:
for symbol in ("m-3m", "-43m"):
    print(f"{symbol}:")
    for entry in diffraction_group_table(symbol):
        group = entry.diffraction_group
        print(f"   b = {np.round(entry.beam_direction, 3)}  {group.symbol:<10} "
              f"BF {group.bright_field_symbol:<5} WP {group.whole_pattern_symbol:<5} "
              f"{'special' if entry.is_special else 'general'}")


Down $[001]$ the two differ in **whole-pattern** symmetry: $4mm$ against $2mm$. Down $[111]$ they
agree in both bright field and whole pattern and cannot be separated. That is why the
determination below is done at $[001]$, and it is the sort of choice the table exists to make
before microscope time is spent.


## 5. The determination, end to end

The controlled pair from section 0: zincblende GaAs and diamond silicon. Identical structure
type, identical site geometry, eight atoms per conventional cell in both — the only difference is
whether the two sublattices carry the same element, and that difference *is* the centre of
symmetry. Kinematic diffraction assigns them the same Laue class and cannot go further.


In [ ]:
def symmetry_config(laue_zones):
    return ConvergentBeamConfig(
        beam_energy_kev=BEAM_KEV,
        convergence_semi_angle_mrad=5.0,
        thickness_angstrom=1000.0,
        disc_samples=45,
        max_index=4,
        g_max_inv_angstrom=1.2,
        max_excitation_error_inv_angstrom=0.005,
        method="bloch",
        laue_zones=laue_zones,
        holz_max_index=20,
        holz_g_max_inv_angstrom=4.8,
        absorption=AbsorptionModel(),
    )


patterns = {}
for phase in (GAAS, SILICON):
    zone = ZoneAxis(np.array([0, 0, 1]), phase=phase)
    patterns[phase.name] = simulate_cbed_pattern(phase, zone, config=symmetry_config((0, 1)))
    pattern = patterns[phase.name]
    print(f"{phase.name}: {len(pattern.discs)} discs drawn, {pattern.beam_set.size} beams coupled "
          f"({int(np.count_nonzero(pattern.beam_set.holz_mask))} of them HOLZ)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.4))
alpha = 5.0
for axis, (name, pattern) in zip(axes, patterns.items()):
    for disc in pattern.discs:
        extent = [disc.centre_mm[0] - disc.radius_mm, disc.centre_mm[0] + disc.radius_mm,
                  disc.centre_mm[1] - disc.radius_mm, disc.centre_mm[1] + disc.radius_mm]
        axis.imshow(np.nan_to_num(disc.intensity, nan=0.0).T ** 0.4, origin="lower",
                    extent=extent, cmap="magma")
    reach = 1.35 * max(np.linalg.norm(d.centre_mm) for d in pattern.discs)
    axis.set_xlim(-reach, reach)
    axis.set_ylim(-reach, reach)
    axis.set_aspect("equal")
    axis.set_title(f"{name}\npredicted {pattern.predicted_diffraction_group().symbol}")
    axis.set_xlabel("detector x (mm)")
axes[0].set_ylabel("detector y (mm)")
fig.tight_layout()
plt.show()


### 5.1 Reading the symmetry back

`symmetry_observations()` tests candidate plane operations against the computed intensities. The
candidates come from the pattern itself: rotations of order 2, 3, 4 and 6, and mirror lines at
the azimuths of the disc centres and their perpendiculars — the only orientations at which a
mirror could permute the discs. The survivors are closed into a group before being named,
because $\{1, R_2, R_3, R_6\}$ is four matrices but a six-fold group.


In [ ]:
for name, pattern in patterns.items():
    observations = pattern.symmetry_observations()
    determination = pattern.determine_point_group()
    print(f"{name}")
    print(f"   predicted diffraction group : {pattern.predicted_diffraction_group().symbol}")
    print(f"   measured bright field       : {observations.bright_field}")
    print(f"   measured whole pattern      : {observations.whole_pattern}")
    print(f"   consistent diffraction group: {determination.diffraction_groups}")
    print(f"   consistent point groups     : {determination.point_groups}")
    print(f"   centrosymmetric             : {determination.is_centrosymmetric}")
    print()
print(patterns[GAAS.name].determine_point_group().describe())
print()

# The decision does not rest on a tolerance: the candidate operations either
# reproduce the pattern or fail outright, so the reading is flat across two
# decades of the threshold.
for tolerance in (0.005, 0.02, 0.05, 0.2):
    read = patterns[GAAS.name].symmetry_observations(tolerance=tolerance)
    print(f"tolerance {tolerance:5.3f} -> BF {read.bright_field:<4} WP {read.whole_pattern}")


The bright-field discs are indistinguishable — both $4mm$. The **whole pattern** separates them:
$2mm$ for the polar structure against $4mm$ for the centric one, which is diffraction group
$4_Rmm_R$ against $4mm1_R$, which is $\{\bar{4}2m, \bar{4}3m\}$ against a centrosymmetric-inclusive
set. And the last block above is the reassuring part: the candidate operations either reproduce
the pattern or fail outright, so the reading is flat across two decades of the threshold. No
tolerance judgement is doing the work.

The control's verdict is left open rather than declared centric, and that is correct: diffraction
group $4mm$ (crystal point group $4mm$, which is acentric) also gives $\mathrm{BF} = \mathrm{WP} =
4mm$. Separating those two needs Buxton's $\pm\mathbf{G}$ observation, which is not implemented —
so the report says so instead of narrowing further than the evidence allows.


### 5.2 The trap, demonstrated

Now the same crystal, the same code, one flag: confine the beam set to the zeroth Laue zone.


In [ ]:
zone = ZoneAxis(np.array([0, 0, 1]), phase=GAAS)
projection = simulate_cbed_pattern(GAAS, zone, config=symmetry_config((0,)))
projected = projection.symmetry_observations(require_holz=False)
print(f"GaAs [001], zeroth Laue zone only: BF = {projected.bright_field}, "
      f"WP = {projected.whole_pattern}")
print(f"   -> {determine_point_group(projected).point_groups}")
print()
zolz = potential_coefficients_inv_angstrom(GAAS, [[2, 2, 0], [4, 0, 0], [2, 0, 0]])
holz = potential_coefficients_inv_angstrom(GAAS, [[1, 1, 1], [3, 1, 1], [1, 1, -1]])
print("why: the ZOLZ potential coefficients of zincblende down [001] are real,")
print("     so the projected structure is centrosymmetric even though the crystal is not.")
print(f"   ZOLZ max |Im nu_g| = {np.max(np.abs(np.imag(zolz))):.2e} 1/A")
print(f"   HOLZ max |Im nu_g| = {np.max(np.abs(np.imag(holz))):.2e} 1/A")


A projection calculation reports the four-fold whole-pattern symmetry of a centrosymmetric
crystal, and the missing centre is invisible. The reason is in the last two numbers: the
zeroth-Laue-zone Fourier coefficients are real, so the structure matrix is *symmetric*, so the
propagator is symmetric, so Friedel's law holds — for the **projected** structure, which really
is centrosymmetric. Admit the first-order Laue zone and the coefficients acquire phases and the
symmetry breaks.

**Higher-order Laue zone interaction is not a refinement here; it is the entire mechanism.** A
symmetry conclusion drawn from a ZOLZ-only calculation is worthless, which is why
`symmetry_observations()` refuses to produce one unless asked a second time, and refuses a
two-beam pattern outright.


In [ ]:
two_beam_pattern = simulate_cbed_pattern(
    GAAS, zone, config=ConvergentBeamConfig(disc_samples=21, beam_energy_kev=BEAM_KEV))
for label, call in (
    ("two-beam pattern", lambda: two_beam_pattern.symmetry_observations()),
    ("projection calculation", lambda: projection.symmetry_observations()),
):
    try:
        call()
    except ValueError as error:
        print(f"{label}:\n   {str(error)[:200]}...\n")


### 5.3 Why the $\pm\mathbf{g}$ relation is not measured here

Buxton's $2_R$ observation compares the $+\mathbf{g}$ and $-\mathbf{g}$ **dark-field** discs, each
recorded with its own reflection at the Bragg condition — two exposures at different specimen
tilts, related by reciprocity. It is *not* a two-fold rotation of a single zone-axis pattern.

Taking it to be one gives a test that quietly fails, and the excitation errors say why:
$s_{-g}(-\boldsymbol{\theta}) - s_g(\boldsymbol{\theta}) = -2 g_z$, which vanishes only in the
zeroth Laue zone. Once higher-order beams are admitted the two-fold is broken for a
centrosymmetric crystal too — and the residual *grows* as the beam set grows, for centric and
acentric structures alike, so it is physics rather than truncation.


In [ ]:
def friedel_residual(phase, window):
    axis = ZoneAxis(np.array([1, 1, 1]), phase=phase)
    selected = beam_set_for_zone(phase, axis, beam_energy_kev=BEAM_KEV, laue_zones=(0, 1, -1),
                                 max_index=16, g_max_inv_angstrom=3.9,
                                 max_excitation_error_inv_angstrom=window,
                                 convergence_semi_angle_mrad=6.0)
    # Symmetrize the truncation so the test measures physics, not the beam list.
    closed = beam_set_from_indices(
        phase, axis,
        np.concatenate([selected.miller_indices, -selected.miller_indices], axis=0),
        beam_energy_kev=BEAM_KEV)
    probe = np.random.default_rng(7).uniform(-6e-3, 6e-3, size=(12, 2))
    result = solve_bloch_waves(closed, np.concatenate([probe, -probe]),
                               thickness_angstrom=1000.0, absorption=AbsorptionModel())
    half = probe.shape[0]
    worst = 0.0
    for index in range(1, closed.size):
        if closed.laue_zone[index] != 0:
            continue
        partner = closed.index_of(-closed.miller_indices[index])
        left = result.intensities[:half, index]
        right = result.intensities[half:, partner]
        spread = float(np.mean(np.abs(left - left.mean())))
        if spread > 0:
            worst = max(worst, float(np.mean(np.abs(right - left))) / spread)
    return closed.size, worst


print(f"  {'window':>8} {'beams':>6}  {'GaAs (acentric)':>16} {'silicon (centric)':>18}")
for window in (0.002, 0.005, 0.012):
    size_a, polar = friedel_residual(GAAS, window)
    size_b, centric_value = friedel_residual(SILICON, window)
    print(f"  {window:8.3f} {size_a:6d}  {polar:16.3f} {centric_value:18.3f}")


Neither column settles toward zero as the beam set grows, and the two stay within a factor of
two or three of each other. A boolean read off that comparison would be right sometimes and wrong
sometimes, so `symmetry_observations()` leaves `friedel_pair_two_fold` to the caller — who may
have made the observation properly, on a $\pm\mathbf{g}$ dark-field pair — rather than reporting
a number it cannot stand behind. The determination does not need it: at $[001]$ the bright-field
and whole-pattern symmetries settle the centre outright, as section 5.1 showed.

There is a second lesson buried in that table. The obvious control — the same two species on the
same lattice, offset by $(	frac12,	frac12,	frac12)$ instead of $(	frac14,	frac14,	frac14)$
— is excellent for the whole-pattern test and *useless* here, because gallium and arsenic are
neighbours in the periodic table and its higher-order reflections, which go as
$f_{\mathrm{Ga}} - f_{\mathrm{As}}$, are nearly extinct. It would have shown a residual near zero
and made the naive test look reliable. A control that is perfect for one measurement can be
uninformative for another.


## 6. What is here, and what is not

**Implemented.** Many-beam Bloch-wave coupling with the beams solved exactly; an absorptive
optical potential from which anomalous absorption emerges rather than being applied; exact HOLZ
line geometry with its sharpness, sensitivity and intersection amplification; the 31 diffraction
groups derived from PyTex's own operator tables together with the point-group table by inversion;
and the determination end to end from a simulated pattern.

**Not implemented, and stated in `describe()` rather than left to be discovered.**

- **Bethe perturbation of weak beams.** Every beam is solved exactly, so a full HOLZ ring costs
  $O(m n^3)$ in earnest. The economy that works is a tighter excitation window, which removes
  beams that were barely coupled — not coarser tilt sampling, which degrades the very fringe
  positions the calculation exists to predict.
- **Absorptive form factors.** The magnitudes are the customary phenomenological ratios; the
  Einstein-model thermal-diffuse integral of Hall and Hirsch, as parametrized by Bird and King,
  is not computed.
- **Buxton's dark-field and $\pm\mathbf{G}$ observations for reflections on symmetry lines.**
  These would separate cases such as $4mm1_R$ from $4mm$ that the two implemented observables
  leave open; the report recommends a second zone axis instead, and names the tool that finds one.
- **Specimen realism.** Perfect parallel-sided slab, column approximation: no wedge, no bending,
  no strain gradient, no surface relaxation, no probe aberration, no inelastic background.

The general lesson is the one section 5.2 makes concrete. The difference between a CBED
simulation that looks right and one that *is* right is rarely in the geometry, which is easy; it
is in whether the beam set samples the crystal or merely its projection.
